# 01 · Portfolio Overview
**Personal Quantitative Research Platform**  
*Last updated: auto-refreshes on every run*

This notebook gives a complete picture of your portfolio:
- Current P&L by position (KRW-converted)
- Core vs Satellite weight breakdown
- Sector (sub_category) breakdown
- Rebalancing alerts (±5pp drift)
- Satellite hard-ceiling guard (25% max)

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})
pd.set_option("display.float_format", "{:,.2f}".format)
print("Libraries loaded ✓")

## 1 · Load Portfolio Snapshot

In [ ]:
from src.data_fetcher import build_portfolio_snapshot

# Tries Google Sheets first, falls back to data/holdings.csv
snap = build_portfolio_snapshot(use_sheets=True)
print(f"Loaded {len(snap)} positions")
snap.head(3)

## 2 · Portfolio Summary

In [ ]:
from src.portfolio import portfolio_summary, compute_weights

snap = compute_weights(snap)
s = portfolio_summary(snap)

print(f"""
╔══════════════════════════════════════════╗
  Portfolio Summary
╠══════════════════════════════════════════╣
  Total Value  :  ₩{s['total_value_krw']:>14,.0f}
               :  ${s['total_value_usd']:>14,.2f}
  Total Cost   :  ₩{s['total_cost_krw']:>14,.0f}
  Unrealised P&L:  ₩{s['total_pnl_krw']:>13,.0f}  ({s['total_pnl_pct']:+.2f}%)
  Positions    :  {s['num_positions']}
  USD/KRW      :  {s['usdkrw']:,.2f}
╠══════════════════════════════════════════╣
  Core Weight  :  {s['core_weight']*100:.1f}%  (₩{s['core_value_krw']:,.0f})
  Satellite    :  {s['satellite_weight']*100:.1f}%  (₩{s['satellite_value_krw']:,.0f})
╚══════════════════════════════════════════╝""")

## 3 · Position Detail Table

In [ ]:
display_cols = [
    "ticker","name","shares","avg_cost","current_price","currency",
    "value_krw","cost_krw","pnl_krw","pnl_pct","weight","category","sub_category"
]

tbl = snap[display_cols].copy()
tbl["pnl_pct"]    = tbl["pnl_pct"].map("{:+.2f}%".format)
tbl["weight"]     = tbl["weight"].map("{:.1%}".format)
tbl["value_krw"]  = tbl["value_krw"].map("₩{:,.0f}".format)
tbl["pnl_krw"]    = tbl["pnl_krw"].map("₩{:+,.0f}".format)

tbl.sort_values("value_krw", ascending=False).reset_index(drop=True)

## 4 · Core / Satellite Breakdown

In [ ]:
from src.portfolio import sector_breakdown

# Bar chart: weights by sub_category, coloured by core/satellite
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: core vs satellite pie
core_sat = snap.groupby("category")["weight"].sum()
axes[0].pie(
    core_sat.values,
    labels=[f"{k}\n{v:.1%}" for k, v in core_sat.items()],
    colors=["#4C72B0","#DD8452"],
    autopct="%1.1f%%", startangle=90, textprops={"fontsize":12}
)
axes[0].set_title("Core / Satellite Split", fontsize=13)

# Right: sub_category weights
sec = sector_breakdown(snap)
colors = ["#4C72B0" if snap[snap.sub_category==s].category.iloc[0]=="core" else "#DD8452"
          for s in sec.sub_category]
axes[1].barh(sec.sub_category, sec.weight * 100, color=colors)
axes[1].set_xlabel("Weight (%)")
axes[1].set_title("Sector Breakdown (blue=core / orange=satellite)", fontsize=12)
for i, (w, pnl) in enumerate(zip(sec.weight, sec.pnl_pct)):
    axes[1].text(w*100 + 0.2, i, f"{pnl:+.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()
print("\nSector Detail:")
sec[["sub_category","num_positions","value_krw","weight","pnl_pct"]]

## 5 · P&L Waterfall

In [ ]:
# Waterfall by position
snap_sorted = snap.sort_values("pnl_krw", ascending=True)

colors = ["#2ecc71" if x >= 0 else "#e74c3c" for x in snap_sorted.pnl_krw]
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(snap_sorted.name, snap_sorted.pnl_krw / 10000, color=colors, edgecolor="white")
ax.axvline(0, color="white", lw=1.5, ls="--")
ax.set_xlabel("Unrealised P&L (₩ 만원)")
ax.set_title("Position P&L Waterfall", fontsize=14, pad=12)
for bar, val in zip(bars, snap_sorted.pnl_krw / 10000):
    ax.text(val + (200 if val >= 0 else -200), bar.get_y() + bar.get_height()/2,
            f"₩{val:+,.0f}만", va="center", ha="left" if val >= 0 else "right", fontsize=9)
plt.tight_layout()
plt.show()

## 6 · Rebalancing Alerts

In [ ]:
from src.portfolio import rebalancing_alerts, satellite_guard

alerts = rebalancing_alerts(snap)
guard  = satellite_guard(snap)

# Satellite guard status
if guard["breach"]:
    print(f"⚠️  SATELLITE BREACH: {guard['satellite_weight']:.1%} > 25% ceiling")
    print(f"   Excess ≈ ₩{guard['excess_krw']:,.0f} needs to be trimmed")
else:
    print(f"✅ Satellite within ceiling: {guard['satellite_weight']:.1%} / 25%")

print()
if alerts.empty:
    print("✅ All positions within ±5pp of target — no rebalancing needed.")
else:
    print(f"⚠️  {len(alerts)} position(s) need attention:")
    display(alerts)